# Params do Kim — métricas completas (T4 / GPU)

Roda o multiclasse `content_ext` com os **parâmetros exatos do Kim** (1000 árvores, lr 0,05,
depth 6, subsample/colsample 0,8, L2=1,0) em **GPU**, nos **dois splits** (aleatório e temporal),
e produz **tudo**: tabela de métricas por classe, **matriz de confusão** e **curvas ROC/PR**.
Referência (nossos params 300/0,3/d8): random **0,9936** / temporal **0,9658**.

**Ative a GPU:** Ambiente de execução → Alterar tipo → **T4 GPU** → Executar tudo.

In [ ]:
# Setup: clona o repo + baixa features (LFS, blindado)
import os
REPO = 'someip-ids-multiclass-contentext'
if not os.path.exists('data/ours_ext/X.npz'):
    if os.path.basename(os.getcwd()) == 'notebooks':
        os.chdir('..')
    elif os.path.isdir(REPO):
        os.chdir(REPO)
    else:
        os.system('apt-get -qq install -y git-lfs >/dev/null 2>&1; git lfs install')
        os.system(f'git clone -q https://github.com/GuilhermeFrick/{REPO}.git')
        os.chdir(REPO); os.system('git lfs pull')
if os.path.exists('data/ours_ext/X.npz') and os.path.getsize('data/ours_ext/X.npz') < 100000:
    os.system('git lfs install; git lfs pull')
import xgboost; print('xgboost', xgboost.__version__, '| cwd:', os.getcwd())

In [ ]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import (classification_report, confusion_matrix, roc_curve, auc,
                             precision_recall_curve, average_precision_score)
from sklearn.preprocessing import label_binarize
from xgboost import XGBClassifier

CLASSES = ['normal','dos','fuzzy','mitm_single','mitm_multi']; N=len(CLASSES); SEED=0
CONTENT_EXT = list(range(12)) + [12, 13, 14, 16]
FILE_COUNTS = [('benign',2193802),('dos',1864530),('fuzzy1',2197113),('fuzzy2',1304154),
               ('fuzzy3',2223650),('mitm_multi',2412529),('mitm_single',2037576)]
OURS_REF = {'random': 0.9936, 'temporal': 0.9658}

X = np.load('data/ours_ext/X.npz')['a'][:, CONTENT_EXT]
y = np.load('data/ours_ext/y_multi.npz')['a']
print('X:', X.shape)

def kim_xgb():  # params do Kim (Tabela 2), em GPU. Sem GPU: troque device='cpu'.
    return XGBClassifier(objective='multi:softprob', num_class=N, n_estimators=1000, max_depth=6,
                         learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0,
                         tree_method='hist', device='cuda', n_jobs=-1, eval_metric='mlogloss')

def report_df(y_true, proba):
    r = classification_report(y_true, proba.argmax(1), target_names=CLASSES, digits=4, output_dict=True)
    df = pd.DataFrame(r).T
    df['support'] = df['support'].astype(int)
    return df.round(4)

def plot_cm(y_true, proba, title):
    cm = confusion_matrix(y_true, proba.argmax(1)); cmn = cm / cm.sum(1, keepdims=True)
    fig, ax = plt.subplots(figsize=(6.5,5.5)); im = ax.imshow(cmn, cmap='Blues', vmin=0, vmax=1)
    ax.set_xticks(range(N)); ax.set_yticks(range(N))
    ax.set_xticklabels(CLASSES, rotation=45, ha='right'); ax.set_yticklabels(CLASSES)
    ax.set_xlabel('Predito'); ax.set_ylabel('Verdadeiro'); ax.set_title(title)
    for i in range(N):
        for j in range(N):
            ax.text(j, i, f'{cmn[i,j]*100:.1f}%', ha='center', va='center', fontsize=8,
                    color='white' if cmn[i,j]>0.5 else 'black')
    fig.colorbar(im, fraction=0.046, pad=0.04); plt.tight_layout(); plt.show()

def plot_curves(y_true, proba, title):
    yb = label_binarize(y_true, classes=range(N))
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(14,5.5))
    for i, name in enumerate(CLASSES):
        fpr, tpr, _ = roc_curve(yb[:,i], proba[:,i]); ar = auc(fpr, tpr)
        s = max(len(fpr)//2000,1); a1.plot(fpr[::s], tpr[::s], lw=1.6, label=f'{name} (AUC={ar:.3f})')
        pr, rc, _ = precision_recall_curve(yb[:,i], proba[:,i]); ap = average_precision_score(yb[:,i], proba[:,i])
        s2 = max(len(rc)//2000,1); a2.plot(rc[::s2], pr[::s2], lw=1.6, label=f'{name} (AP={ap:.3f})')
    a1.plot([0,1],[0,1],'k--',lw=0.7); a1.set_xlabel('Falso Positivo'); a1.set_ylabel('Verdadeiro Positivo')
    a1.set_title(f'ROC OvR — {title}'); a1.legend(fontsize=8, loc='lower right')
    a2.set_xlabel('Recall'); a2.set_ylabel('Precisão'); a2.set_title(f'Precisão-Recall OvR — {title}')
    a2.legend(fontsize=8, loc='lower left'); plt.tight_layout(); plt.show()

In [ ]:
# Treina os dois modelos (params do Kim) e guarda as probabilidades
Xtr, Xte, ytr, y_rand = train_test_split(X, y, test_size=0.30, random_state=SEED, stratify=y)
m_rand = kim_xgb(); m_rand.fit(Xtr, ytr); p_rand = m_rand.predict_proba(Xte)
print('aleatório OK')

tr, te, s = [], [], 0
for _, c in FILE_COUNTS:
    cut = s + int(c*0.7); tr.append(np.arange(s, cut)); te.append(np.arange(cut, s+c)); s += c
itr, ite = np.concatenate(tr), np.concatenate(te); y_temp = y[ite]
m_temp = kim_xgb(); m_temp.fit(X[itr], y[itr]); p_temp = m_temp.predict_proba(X[ite])
print('temporal OK')

In [ ]:
# Comparação macro-F1: params Kim vs nossos params
from sklearn.metrics import f1_score
mf = {'random': f1_score(y_rand, p_rand.argmax(1), average='macro'),
      'temporal': f1_score(y_temp, p_temp.argmax(1), average='macro')}
display(pd.DataFrame({'params Kim (1000/0.05/d6/sub0.8)': mf,
                      'nossos params (300/0.3/d8)': OURS_REF}).round(4))

## Split TEMPORAL (honesto) — métricas, matriz de confusão, curvas

In [ ]:
print('=== TEMPORAL (params Kim) — métricas por classe ===')
display(report_df(y_temp, p_temp))

In [ ]:
plot_cm(y_temp, p_temp, 'Matriz de Confusão — TEMPORAL (params Kim)')

In [ ]:
plot_curves(y_temp, p_temp, 'TEMPORAL (params Kim)')

## Split ALEATÓRIO (otimista) — métricas e matriz de confusão (comparação)

In [ ]:
print('=== ALEATÓRIO (params Kim) — métricas por classe ===')
display(report_df(y_rand, p_rand))
plot_cm(y_rand, p_rand, 'Matriz de Confusão — ALEATÓRIO (params Kim)')

## Leitura

- Com os params **mais regularizados** do Kim, os números ficam **iguais aos nossos**
  (random ≈ 0,994 / temporal ≈ 0,967) → resultado **robusto a hiperparâmetros**.
- Na matriz de confusão **temporal**, a única classe que sofre é **mitm_multi** (relay) — o resto
  fica > 0,99. No **aleatório** tudo fica quase perfeito (efeito do vazamento temporal).
- Número honesto = **temporal**; generalização a ataque novo = **zero-day ~0,60** (`someip-ids-benchmark`).